# TS/TL 2번 조합 누수 점검

해당 Notebook 파일은 TS/TL 2번 조합에 포함되는 이미지가 그 외 조합(1, 3~8번 zip) 폴더에도 포함되어 있는지 점검합니다.

같은 파일명 체계를 사용하므로 파일명의 K-Code 부분을 기준으로 **1) K-Code 파일명이 전부 동일한 경우**, **2) 나열 순서와 상관 없이 알약 구성이 동일한 경우**를 탐색합니다. 1) 또는 2)에서 탐색된 이미지가 있을 경우 **픽셀 유사도를 기준**으로 정렬하여 시각화합니다.

In [ ]:
!pip install -q scikit-image

In [ ]:
import os, re, json, io, random, zipfile, tarfile
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

# ── 경로 설정 ──────────────────────────────────────────────
TEAM_DIR    = "/content/drive/MyDrive"
WORK_DIR    = "/content/aihub_work"
os.makedirs(WORK_DIR, exist_ok=True)

TL_TAR      = os.path.join(TEAM_DIR, "TL_2_조합.tar")
TS_TAR      = os.path.join(TEAM_DIR, "TS_2_조합.tar")

# ── N종 18개 타겟 K-code ──────────────────────────────────
TARGET_N = {
    27653, 23223, 6563, 6192, 31705, 18110, 10221, 21026,
    5094, 44199, 23203, 10224, 33878, 22627, 29871,
    4378, 5886, 12420,
}

# ── TL tar 추출 ───────────────────────────────────────────
with tarfile.open(TL_TAR, 'r') as t:
    member     = t.getmembers()[0]
    t.extract(member, path=WORK_DIR, filter='data')
    part0_path = os.path.join(WORK_DIR, member.name)
print(f"TL 추출 완료: {part0_path}")

# ── TS tar 분할 병합 ──────────────────────────────────────
ts_zip_path = os.path.join(WORK_DIR, "TS_2_merged.zip")
if not os.path.exists(ts_zip_path):
    print("TS 분할 병합 중...")
    parts = []
    with tarfile.open(TS_TAR, 'r') as t:
        for m in sorted(t.getmembers(), key=lambda x: x.name):
            parts.append(t.extractfile(m).read())
    with open(ts_zip_path, 'wb') as f:
        f.write(b''.join(parts))
    print(f"병합 완료: {os.path.getsize(ts_zip_path)/1024/1024:.1f} MB")
else:
    print(f"TS zip 이미 존재")


In [ ]:
import os, re, io, zipfile, tarfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

# ── 경로 설정 ──────────────────────────────────────────────
MY_DIR   = "/content/drive/MyDrive"
TEAM_DIR = "/content/drive/MyDrive"
WORK_DIR = "/content/aihub_work"
os.makedirs(WORK_DIR, exist_ok=True)

TL_TAR = os.path.join(MY_DIR, "TL_2_조합.tar")
TS_TAR = os.path.join(MY_DIR, "TS_2_조합.tar")

TARGET_NUMS = [1, 3, 4, 5, 6, 7, 8]   # 2 제외 (2와 대조할 대상)

# ── TL_2 tar 추출 ─────────────────────────────────────────
with tarfile.open(TL_TAR, 'r') as t:
    member = t.getmembers()[0]
    t.extract(member, path=WORK_DIR, filter='data')
    part0_path = os.path.join(WORK_DIR, member.name)
print(f"TL 추출 완료: {part0_path}")

# ── TS_2 tar 분할 병합 ────────────────────────────────────
ts_zip_path = os.path.join(WORK_DIR, "TS_2_merged.zip")
if not os.path.exists(ts_zip_path):
    print("TS 분할 병합 중...")
    parts = []
    with tarfile.open(TS_TAR, 'r') as t:
        for m in sorted(t.getmembers(), key=lambda x: x.name):
            parts.append(t.extractfile(m).read())
    with open(ts_zip_path, 'wb') as f:
        f.write(b''.join(parts))
    print(f"병합 완료: {os.path.getsize(ts_zip_path)/1024/1024:.1f} MB")
else:
    print("TS zip 이미 존재")

# ── 1. 파일명 파싱 함수 ──────────────────────────────────
def parse_filename(fn):
    """
    'K-001900-003544-016548_0_2_0_2_70_000_200.png'
    -> combo_str='K-001900-003544-016548', kcodes=(1900,3544,16548), lat=70
    """
    base = os.path.basename(fn)
    m = re.match(r'^(K-[\d-]+)_0_2_0_2_(\d+)_000_200', base)
    if not m:
        return None
    combo_str, lat = m.group(1), int(m.group(2))
    kcodes = tuple(int(x) for x in combo_str.split('-')[1:])
    return combo_str, kcodes, lat

# ── 2. TS_2(타겟) 인덱스 구축 ─────────────────────────────
target_exact_index = defaultdict(list)
target_set_index   = defaultdict(list)

with zipfile.ZipFile(ts_zip_path, 'r') as z:
    target_names = [f for f in z.namelist()
                     if f.lower().endswith('.png') and 'index' not in f]

for fn in target_names:
    parsed = parse_filename(fn)
    if not parsed:
        continue
    combo_str, kcodes, lat = parsed
    target_exact_index[combo_str].append((fn, lat))
    target_set_index[frozenset(kcodes)].append((fn, lat, combo_str))

print(f"TS_2 타겟 이미지: {len(target_names)}장")
print(f"고유 exact combo: {len(target_exact_index)}개")
print(f"고유 kcode set  : {len(target_set_index)}개")

# ── 3. TS_1,3~8 순회하며 매칭 확인 ────────────────────────
method1_matches = []
method2_matches = []

for num in TARGET_NUMS:
    ts_path = os.path.join(TEAM_DIR, f"TS_{num}_조합.zip")
    if not os.path.exists(ts_path):
        print(f"  [SKIP] TS_{num}_조합.zip 없음")
        continue

    print(f"TS_{num}_조합.zip 스캔 중...")
    with zipfile.ZipFile(ts_path, 'r') as z:
        cand_names = [f for f in z.namelist()
                       if f.lower().endswith('.png') and 'index' not in f]

        for fn in cand_names:
            parsed = parse_filename(fn)
            if not parsed:
                continue
            combo_str, kcodes, lat = parsed
            kset = frozenset(kcodes)

            if combo_str in target_exact_index:
                for t_fn, t_lat in target_exact_index[combo_str]:
                    method1_matches.append({
                        "cand_zip": num, "cand_fn": fn, "cand_lat": lat,
                        "target_fn": t_fn, "target_lat": t_lat,
                        "same_latitude": (lat == t_lat),
                    })

            if kset in target_set_index:
                for t_fn, t_lat, t_combo in target_set_index[kset]:
                    if t_combo == combo_str:
                        continue
                    method2_matches.append({
                        "cand_zip": num, "cand_fn": fn, "cand_lat": lat,
                        "target_fn": t_fn, "target_lat": t_lat,
                        "same_latitude": (lat == t_lat),
                    })

print(f"\n{'='*55}")
print(f"방법 1 (K-code 순서까지 동일): {len(method1_matches)}건")
print(f"방법 2 (K-code 종류만 동일)  : {len(method2_matches)}건")
print(f"{'='*55}")

# ── 4. 결과 요약 출력 ─────────────────────────────────────
print("\n[방법 1] 순서 동일 매칭 (위도 동일/다름 여부 포함)")
for m in method1_matches[:30]:
    flag = "동일위도" if m["same_latitude"] else "다른위도"
    print(f"  TS_{m['cand_zip']}: {m['cand_fn']} (lat={m['cand_lat']}) "
          f"<-> TS_2: {m['target_fn']} (lat={m['target_lat']})  [{flag}]")
if len(method1_matches) > 30:
    print(f"  ... 외 {len(method1_matches)-30}건")

print("\n[방법 2] 종류만 동일 매칭 (순서 무관)")
for m in method2_matches[:30]:
    print(f"  TS_{m['cand_zip']}: {m['cand_fn']} (lat={m['cand_lat']}) "
          f"<-> TS_2: {m['target_fn']} (lat={m['target_lat']})")
if len(method2_matches) > 30:
    print(f"  ... 외 {len(method2_matches)-30}건")

# ── 5. 픽셀 유사도 계산 (방법1+2 매칭 대상 전체) ─────────
from skimage.metrics import structural_similarity as ssim

all_pairs = method1_matches + method2_matches

cand_zip_handles = {}
for num in TARGET_NUMS:
    p = os.path.join(TEAM_DIR, f"TS_{num}_조합.zip")
    if os.path.exists(p):
        cand_zip_handles[num] = zipfile.ZipFile(p, 'r')

target_zip_handle = zipfile.ZipFile(ts_zip_path, 'r')

def load_gray(zip_handle, path, size=(488, 640)):
    img_bytes = zip_handle.read(path)
    img = Image.open(io.BytesIO(img_bytes)).convert("L")
    img = img.resize(size)
    return np.array(img)

print(f"\n픽셀 유사도 계산 중... (총 {len(all_pairs)}쌍)")
similarity_results = []

for i, m in enumerate(all_pairs):
    try:
        cand_img = load_gray(cand_zip_handles[m["cand_zip"]], m["cand_fn"])
        targ_img = load_gray(target_zip_handle, m["target_fn"])
        score = ssim(cand_img, targ_img)
        similarity_results.append({**m, "similarity": score})
    except Exception as e:
        print(f"  [ERROR] {m['cand_fn']}: {e}")
    if i % 20 == 0:
        print(f"  [{i+1}/{len(all_pairs)}]", end="\r")

similarity_results.sort(key=lambda x: x["similarity"], reverse=True)

print(f"\n\n{'='*55}")
print("픽셀 유사도 순위 (상위 20개)")
print(f"{'='*55}")
for r in similarity_results[:20]:
    print(f"  {r['similarity']*100:5.1f}%  |  TS_{r['cand_zip']}: {r['cand_fn'][:40]:40s} "
          f"<-> TS_2: {r['target_fn'][:40]}")

# ── 6. 상위 N개 시각화 ────────────────────────────────────
N_VIZ = 10
top_pairs = similarity_results[:N_VIZ]

fig, axes = plt.subplots(N_VIZ, 2, figsize=(8, N_VIZ * 3))
for i, r in enumerate(top_pairs):
    cand_bytes = cand_zip_handles[r["cand_zip"]].read(r["cand_fn"])
    targ_bytes = target_zip_handle.read(r["target_fn"])
    cand_img = Image.open(io.BytesIO(cand_bytes)).convert("RGB")
    targ_img = Image.open(io.BytesIO(targ_bytes)).convert("RGB")

    axes[i, 0].imshow(cand_img)
    axes[i, 0].set_title(f"TS_{r['cand_zip']} (lat={r['cand_lat']})\n{os.path.basename(r['cand_fn'])}",
                           fontsize=7)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(targ_img)
    axes[i, 1].set_title(f"TS_2 (lat={r['target_lat']}) — 유사도 {r['similarity']*100:.1f}%\n"
                           f"{os.path.basename(r['target_fn'])}", fontsize=7)
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

# ── 정리 ──────────────────────────────────────────────────
for h in cand_zip_handles.values():
    h.close()
target_zip_handle.close()

print(f"\n최종 요약")
print(f"  방법1(순서 동일) 매칭: {len(method1_matches)}건")
print(f"  방법2(종류만 동일) 매칭: {len(method2_matches)}건")
print(f"  유사도 90% 이상 쌍: {sum(1 for r in similarity_results if r['similarity'] > 0.9)}건")